In [ ]:
# auto_stop_fulln.ipynb -- overnight sentinel. Run this on EVERY pod before
# sleeping. It watches the shared out_dir; once every FULL-N (n=2000) combo is
# terminal (done or failed), it SIGTERMs local training (the supervisor
# self-revokes its lease) and STOPS THIS RUNPOD POD -- billing ends, the
# /workspace volume persists. Small-n combos picked up during the drain tail
# are interrupted safely: checkpoints resume later and reconcile refunds the
# attempt. Morning plan: start 2-3 pods, run eval_champions -> prep-select ->
# prep-prune, then resume the champion tail on the cheap subset.
#
# FIRST TIME: pick ONE pod, set TEST_SHUTDOWN=True and run -- it executes the
# exact stop path immediately (kills training cleanly, stops the pod). Confirm
# in the RunPod console that the pod stopped, start it again, resume training
# (a plain training.ipynb run; the startup cleanup handles everything), then
# roll the sentinel out everywhere with TEST_SHUTDOWN=False.
import copy, json, os, shutil, signal, subprocess, sys, time
from pathlib import Path

TEST_SHUTDOWN = False       # True = full dress rehearsal NOW: stop training,
                            # revoke lease, stop THIS pod. No waiting.
REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
POLL_SECONDS = 120
HARD_DEADLINE_HOURS = 12    # stop the pod after this many hours NO MATTER WHAT
                            # (safety net so a stuck combo can't bill all night)

# ---- preflight: fail LOUDLY now, not at 5am --------------------------------
pod_id = os.environ.get('RUNPOD_POD_ID')
ctl = shutil.which('runpodctl')
print(f'preflight: RUNPOD_POD_ID={pod_id or "MISSING"}  runpodctl={ctl or "MISSING"}')
if not pod_id or not ctl:
    print('!! preflight FAILED: this pod cannot stop itself. Training would be')
    print('   stopped but the pod would keep billing while idle. Fix before')
    print('   sleeping, or plan to stop pods from the RunPod console.')

if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review.sweep.config import SweepConfig

root = Path(REPO) / OUT_DIR
cfg = SweepConfig.load(f'{REPO}/VICReg_review/sweep/sweep.yaml')
combos = list(cfg.iter_combos())
_counts = [int(c.train_games) for c in combos]
FULL_N = 0 if any(n <= 0 for n in _counts) else max(_counts)
fulln_ids = [c.combo_id for c in combos if int(c.train_games) == FULL_N]
print(f'sentinel: {len(fulln_ids)} FULL-N combos to watch; poll={POLL_SECONDS}s; '
      f'hard deadline={HARD_DEADLINE_HOURS}h; TEST_SHUTDOWN={TEST_SHUTDOWN}')


def _terminal(cid):
    d = root / cid
    if (d / 'done.json').exists() or (d / 'failed.json').exists():
        return True
    try:
        return json.loads((d / 'vicreg_review_h5_manifest.json')
                          .read_text(encoding='utf-8')).get('status') == 'done'
    except Exception:
        return False


def remaining():
    return [cid for cid in fulln_ids if not _terminal(cid)]


def stop_everything(reason):
    print(f'sentinel: {reason} -- stopping training on this pod', flush=True)
    subprocess.run(['pkill', '-f', 'sweep/supervisor.py'])   # SIGTERM: lease self-revokes
    time.sleep(10)
    subprocess.run(['pkill', '-9', '-f', 'sweep/worker.py'])
    if pod_id and ctl:
        print(f'sentinel: runpodctl stop pod {pod_id}', flush=True)
        r = subprocess.run(['runpodctl', 'stop', 'pod', pod_id],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr, flush=True)
        if r.returncode != 0:
            print('!! runpodctl stop FAILED -- pod is idle but still billing; '
                  'stop it from the RunPod console.', flush=True)
    else:
        print('!! cannot self-stop: training stopped, but STOP THE POD from '
              'the console.', flush=True)


if TEST_SHUTDOWN:
    stop_everything('TEST_SHUTDOWN dress rehearsal')
    print('test issued: within ~a minute this pod should show Stopped in the '
          'RunPod console. Restart it afterwards and re-run training.ipynb '
          '(startup cleanup + attempt refund make the interruption free).')
else:
    deadline = time.time() + HARD_DEADLINE_HOURS * 3600
    while True:
        left = remaining()
        if not left:
            stop_everything('FULL-N group fully terminal')
            break
        if time.time() >= deadline:
            stop_everything(f'hard deadline {HARD_DEADLINE_HOURS}h reached '
                            f'({len(left)} FULL-N combos still open)')
            break
        print(f'{time.strftime("%H:%M:%S")} FULL-N remaining: {len(left)}'
              + (f'  (next few: {", ".join(left[:3])})' if len(left) <= 6 else ''),
              flush=True)
        time.sleep(POLL_SECONDS)
